# 11 — Held-out test set & real-world evaluation

**Purpose:** close the methodological gap flagged in `project-no-held-out-test-set` memory.

Three evaluation points:

1. **Val set** (0.750 macro F1) — what's in `models/runs/v1_2026-05-16/baselines.json`
2. **Real-world 264 manual labels** (0.630 weighted F1) — atlas-ed-data inference output, weeks 1-13
3. **Held-out test** (issues 88-102, never seen in training) — built here

This notebook also covers per-issue F1 (temporal stability), confusion matrix worked errors, and per-source fairness — together they address AM2 criteria K20, S22, S24 + distinction S9/K14.


In [ ]:
import sys, json
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score, accuracy_score,
    ConfusionMatrixDisplay,
)
from sentence_transformers import SentenceTransformer
import matplotlib.pyplot as plt

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

MODEL_DIR = ROOT / "models" / "runs" / "v1_2026-05-16"
CLASSIFIER_PATH = MODEL_DIR / "classifier.joblib"
TRAIN_CSV = ROOT / "data" / "processed" / "train.csv"   # has new_theme label col
VAL_CSV = ROOT / "data" / "processed" / "val.csv"

with open(MODEL_DIR / "run_metadata.json") as f:
    run_meta = json.load(f)
print(f"Model: {run_meta['variant']}")
print(f"Val macro F1 (baseline): {run_meta['metrics']['val_macro_f1']}")
print(f"Real-world weighted F1 (264 labels): {run_meta['metrics']['real_world_weighted_f1']}")


## Step 1 — Build the held-out test set (issues 105–113, in-notebook)

In [ ]:
# Build the held-out test set IN-NOTEBOOK from the raw extracted newsletters,
# using the SAME cleaning/preprocessing functions training used (s01 + s02).
# One source of truth -> no train/serve skew. No s03 split (the whole set is test).
from src.training_data import s01_clean as s01
from src.training_data import s02_preprocess as s02
from src.training_data.s03_split import VALID_THEMES

RAW_CSV = ROOT / "data" / "interim" / "extracted_newsletters.csv"   # <-- source (now holds issues 1-113)
HELD_OUT_ISSUES = range(105, 114)   # 105-113 inclusive = 9 issues, published AFTER training cutoff (issue 104 = 6 Mar 2026)

df = pd.read_csv(RAW_CSV)
df["newsletter_number"] = pd.to_numeric(df["newsletter_number"], errors="coerce")
df = df[df["newsletter_number"].isin(HELD_OUT_ISSUES)].copy()
print(f"Raw held-out items (105-113): {len(df)} across {df['newsletter_number'].nunique()} issues")

# --- s01 clean (identical to training) ---
obj_cols = [c for c in df.columns if df[c].dtype == object]
for c in obj_cols:
    df[c] = s01.clean_series(df[c])
df = s01.fix_artifacts(df, obj_cols)
df = s01.drop_missing_essentials(df)
df = s01.deduplicate(df)

# --- s02 preprocess (identical to training): creates new_theme (the label), organisation, item_type, text features ---
df = s02.add_themes(df)
df = s02.add_domain_and_org(df)
df["item_type"] = df.apply(s02.classify_item_type, axis=1)
df = s02.add_text_features(df)

# --- keep only the model's target classes; surface what gets dropped ---
test_df = df[df["new_theme"].isin(VALID_THEMES)].copy()
dropped = df[~df["new_theme"].isin(VALID_THEMES)]
print(f"\nHeld-out TEST set: {len(test_df)} items in model classes  ({len(dropped)} dropped)")
if len(dropped):
    print("DROPPED (check — 'Updates from PI ...' sections don't map to update_from_pi):")
    print(dropped["new_theme"].value_counts().to_string())
print("\nTest label distribution (new_theme):")
print(test_df["new_theme"].value_counts().to_string())


### Scope, statistical power, and claims

**What this set is.** Issues **105–113** (9 newsletters, ~120 items after preprocessing), published *after* the training cutoff (issue 104 = 6 Mar 2026; 105 = 13 Mar 2026). The model never saw them and they drove no tuning decision — a genuine **temporal** held-out test, unlike the val set (which was used for model selection and is overfit-to).

**Read the CIs, not the point estimate.**
- n ≈ 120 → the **95% bootstrap CI on macro-F1 is wide** (expect roughly ±0.05–0.10).
- Per-class F1 for small classes (n < 10) is **unstable** — always show support (n).
- 9 consecutive issues is an *early* read on temporal drift, not a definitive trend.

**What we CAN say:** the first leakage-free macro-F1 with its CI; the per-issue F1 trend (105→113) as early temporal-stability evidence; whether it's broadly in line with, or below, val 0.750.

**What we must NOT say:** don't quote a bare point estimate; don't make strong per-class claims at tiny n; don't treat this as apples-to-apples with val 0.750 (val was tuned on — a lower held-out number is *expected* and is the point); don't generalise beyond newsletter-style items; 9 issues ≠ a robust drift study.

**Note on dropped items:** the build cell drops sections whose `new_theme` isn't a model class. The `"Updates from PI ⟨name⟩"` headers currently fall through (s02 doesn't map them to `update_from_pi`). Decide whether to (a) keep the test on the 6 substantive content classes, or (b) extend `s02.add_themes` to map the PI/programme sections back in.


In [ ]:
# Sanity check — no URL overlap with train/val.
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
overlap_train = set(test_df.get('link', test_df.get('url', []))) & set(train_df.get('link', train_df.get('url', [])))
overlap_val = set(test_df.get('link', test_df.get('url', []))) & set(val_df.get('link', val_df.get('url', [])))
print(f"URL overlap with train: {len(overlap_train)}  (must be 0)")
print(f"URL overlap with val:   {len(overlap_val)}    (must be 0)")


In [ ]:
# Label distribution test vs train.
print('=== Test label distribution ===')
print(test_df['new_theme'].value_counts())
print()
print('=== Train label distribution (for comparison) ===')
print(train_df['target'].value_counts() if 'target' in train_df else train_df['new_theme'].value_counts())


## Step 2 — Run production model on held-out test

In [ ]:
# Load the production model (SBERT no-meta + LogReg).
classifier = joblib.load(CLASSIFIER_PATH)
encoder = SentenceTransformer(run_meta['embedding_model'])
print(f"Loaded classifier + encoder. Model classes ({len(classifier.classes_)}): {list(classifier.classes_)}")

# The production model is a 6-class model — it cannot output update_from_pi /
# update_from_programme. Restrict the test to classes the model can actually
# predict, so the score reflects discrimination, not architectural impossibility.
n_before = len(test_df)
test_df = test_df[test_df['new_theme'].isin(classifier.classes_)].copy()
print(f"Test restricted to model classes: {len(test_df)} items (dropped {n_before - len(test_df)} in non-model classes)")

# Title-only input (matches training).
test_titles = test_df['title'].fillna('').astype(str).tolist()
test_embeddings = encoder.encode(test_titles, show_progress_bar=True)
print(f"Encoded {len(test_embeddings)} items, dim={test_embeddings.shape[1]}")


In [ ]:
# Predict.
y_pred = classifier.predict(test_embeddings)
y_pred_proba = classifier.predict_proba(test_embeddings)

# Label is already standardised to the model's classes by s02.add_themes -> new_theme.
# No LABEL_MAP needed. Flag any test class the model was never trained on.
y_true = test_df['new_theme']
unknown = set(y_true) - set(classifier.classes_)
print(f"Test classes not in model.classes_ (will always be wrong if any): {unknown or 'none'}")


## Step 3 — Headline metrics

In [ ]:
macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
weighted_f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
top1_acc = accuracy_score(y_true, y_pred)

# Top-2 accuracy.
classes = classifier.classes_
top2_idx = np.argsort(-y_pred_proba, axis=1)[:, :2]
top2_preds = classes[top2_idx]
top2_acc = np.mean([yt in tp for yt, tp in zip(y_true, top2_preds)])

print(f"Held-out test (n={len(test_df)})")
print(f"  Macro F1:    {macro_f1:.3f}  (val baseline: {run_meta['metrics']['val_macro_f1']})")
print(f"  Weighted F1: {weighted_f1:.3f}  (real-world 264-label baseline: {run_meta['metrics']['real_world_weighted_f1']})")
print(f"  Top-1 acc:   {top1_acc:.3f}")
print(f"  Top-2 acc:   {top2_acc:.3f}  (val baseline top-2: {run_meta['metrics']['real_world_top2_accuracy']})")


In [ ]:
# Bootstrap 95% CI on macro-F1 — with n~120 the point estimate is noisy, so report the interval.
def bootstrap_macro_f1(y_true, y_pred, n=2000, seed=42):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    rng = np.random.default_rng(seed)
    idx = np.arange(len(y_true))
    scores = [f1_score(y_true[s], y_pred[s], average="macro", zero_division=0)
              for s in (rng.choice(idx, len(idx), replace=True) for _ in range(n))]
    return float(np.mean(scores)), float(np.percentile(scores, 2.5)), float(np.percentile(scores, 97.5))

boot_mean, ci_lo, ci_hi = bootstrap_macro_f1(np.asarray(y_true), y_pred)
print(f"Held-out macro-F1 = {boot_mean:.3f}  (95% bootstrap CI [{ci_lo:.3f}, {ci_hi:.3f}], n={len(y_true)})")
print("Report the INTERVAL, not just the point estimate — see the scope/claims note above.")


## Step 4 — Per-class breakdown

In [ ]:
print(classification_report(y_true, y_pred, zero_division=0))


## Step 5 — Confusion matrix

In [ ]:
labels = sorted(set(y_true) | set(y_pred))
cm = confusion_matrix(y_true, y_pred, labels=labels)
fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(cm, display_labels=labels)
disp.plot(ax=ax, cmap='Blues', values_format='d', xticks_rotation=45)
ax.set_title('Held-out test — confusion matrix')
plt.tight_layout()
plt.savefig(ROOT / 'outputs' / 'held_out_confusion_matrix.png', dpi=120)
plt.show()


## Step 6 — Worked errors (off-diagonal cells with ≥3 examples)

In [ ]:
# For each off-diagonal (true, pred) pair with ≥3 examples, print sample titles.
test_df['_y_true'] = y_true.values
test_df['_y_pred'] = y_pred
test_df['_correct'] = test_df['_y_true'] == test_df['_y_pred']

errors = test_df[~test_df['_correct']]
pair_counts = errors.groupby(['_y_true', '_y_pred']).size().reset_index(name='n')
pair_counts = pair_counts[pair_counts['n'] >= 3].sort_values('n', ascending=False)

print(f"Off-diagonal pairs with ≥3 errors: {len(pair_counts)}\n")
for _, row in pair_counts.iterrows():
    true_class, pred_class, n = row['_y_true'], row['_y_pred'], row['n']
    print(f"--- {true_class} → predicted {pred_class} ({n} errors) ---")
    sample = errors[(errors['_y_true'] == true_class) & (errors['_y_pred'] == pred_class)].head(3)
    for _, r in sample.iterrows():
        print(f"  '{(r.get('title') or '')[:100]}'")
    print()


## Step 7 — Per-issue F1 (temporal stability)

In [ ]:
# Per-issue macro F1 — does the model hold up over time?
per_issue = []
for issue, g in test_df.groupby('newsletter_number'):
    if len(g) < 5:
        continue
    f1 = f1_score(g['_y_true'], g['_y_pred'], average='macro', zero_division=0)
    per_issue.append({'issue': issue, 'n': len(g), 'macro_f1': f1})

per_issue_df = pd.DataFrame(per_issue).sort_values('issue')
print(per_issue_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(per_issue_df['issue'], per_issue_df['macro_f1'], marker='o')
ax.axhline(run_meta['metrics']['val_macro_f1'], color='grey', linestyle='--', label=f"Val baseline ({run_meta['metrics']['val_macro_f1']})")
ax.set_xlabel('Newsletter issue number')
ax.set_ylabel('Macro F1')
ax.set_title('Held-out per-issue F1 — temporal stability')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(ROOT / 'outputs' / 'held_out_per_issue_f1.png', dpi=120)
plt.show()


## Conclusion

Three-point evaluation arc:

| Point | Macro F1 / metric | Source |
|---|---|---|
| Val | 0.750 | Single split of issues 1–104, **used for model selection** |
| Real-world 264 labels | 0.630 weighted | Atlas-ed inference output |
| **Held-out test (issues 105–113)** | (computed above, ~0.50 + 95% CI) | **Fully unseen — published after training cutoff** |

**Honest caveat:** the val set was used for the SHAP-driven model switch + 5+ other decisions, so 0.750 is overfit. The held-out test (105–113, post-6-Mar-2026) is the first uncontaminated number — and the drop from 0.75 → ~0.50 is itself the key finding: it quantifies how much the val score was inflated and is direct evidence of concept drift / overfitting.

For AM2 distinction (S9/K14): report the held-out macro-F1 **with its bootstrap CI** (n~116, so the interval is wide — say the interval, not the point). The per-issue F1 plot shows temporal stability across 9 consecutive weeks. A much-lower held-out number justifies the retraining trigger criteria in [`docs/decisions/model_v1_state_and_retraining_plan.md`](../docs/decisions/model_v1_state_and_retraining_plan.md). See the scope/claims note in Step 1 for what can and cannot be claimed.